# SQL for Python Users — Practical Workshop

A hands-on bridge from Python/pandas to relational databases. Uses SQLite, which is included with Python: no server or account required.

**How to use:** run the setup, attempt each exercise before opening its solution, and experiment by changing the queries. The dataset is fictional.

## The mental model

In pandas you commonly load data into DataFrames and transform them. SQL is **declarative**: you describe the rows and columns you want, and the database works out how to retrieve them.

| SQL | Approximate pandas equivalent |
|---|---|
| `SELECT a, b` | `df[["a", "b"]]` |
| `WHERE x > 3` | `df[df.x > 3]` |
| `ORDER BY x` | `df.sort_values("x")` |
| `GROUP BY` | `df.groupby(...)` |
| `JOIN` | `pd.merge(...)` |
| `LIMIT 5` | `df.head(5)` |


## 0. Setup — create a database

A database contains tables. Each table has columns (with defined types) and rows. We'll create four related tables: customers, orders, products and order_items.

In [3]:
import sqlite3
import pandas as pd

con = sqlite3.connect(":memory:")
con.execute("PRAGMA foreign_keys = ON;")

con.executescript('''
CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    city TEXT NOT NULL,
    signup_date TEXT NOT NULL
);
CREATE TABLE products (
    product_id INTEGER PRIMARY KEY,
    product_name TEXT NOT NULL,
    category TEXT NOT NULL,
    unit_price REAL NOT NULL
);
CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY,
    customer_id INTEGER NOT NULL,
    order_date TEXT NOT NULL,
    status TEXT NOT NULL,
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
);
CREATE TABLE order_items (
    order_item_id INTEGER PRIMARY KEY,
    order_id INTEGER NOT NULL,
    product_id INTEGER NOT NULL,
    quantity INTEGER NOT NULL,
    FOREIGN KEY (order_id) REFERENCES orders(order_id),
    FOREIGN KEY (product_id) REFERENCES products(product_id)
);

INSERT INTO customers VALUES
(1, 'Aisha Khan', 'London', '2024-01-10'),
(2, 'Ben Carter', 'Manchester', '2024-02-03'),
(3, 'Chloe Smith', 'London', '2024-02-18'),
(4, 'Daniel Reed', 'Bristol', '2024-03-02'),
(5, 'Eva Patel', 'Leeds', '2024-03-20');

INSERT INTO products VALUES
(1, 'Laptop', 'Electronics', 900.0),
(2, 'Mouse', 'Electronics', 25.0),
(3, 'Desk', 'Furniture', 220.0),
(4, 'Chair', 'Furniture', 130.0),
(5, 'Notebook', 'Stationery', 4.5);

INSERT INTO orders VALUES
(101, 1, '2024-04-01', 'shipped'),
(102, 2, '2024-04-03', 'shipped'),
(103, 1, '2024-04-15', 'processing'),
(104, 3, '2024-05-02', 'shipped'),
(105, 4, '2024-05-10', 'cancelled'),
(106, 5, '2024-05-12', 'shipped'),
(107, 2, '2024-06-01', 'shipped');

INSERT INTO order_items VALUES
(1,101,1,1), (2,101,2,2), (3,102,3,1), (4,102,5,5),
(5,103,4,2), (6,104,1,1), (7,104,4,1), (8,105,3,1),
(9,106,5,10), (10,107,2,1), (11,107,4,1);
''')

def query(sql, params=None):
    return pd.read_sql_query(sql, con, params=params)

query("SELECT * FROM customers;")


,customer_id,name,city,signup_date
0,1,Aisha Khan,London,2024-01-10
1,2,Ben Carter,Manchester,2024-02-03
2,3,Chloe Smith,London,2024-02-18
3,4,Daniel Reed,Bristol,2024-03-02
4,5,Eva Patel,Leeds,2024-03-20


## 1. SELECT, WHERE, ORDER BY, LIMIT

The basic query shape:

```sql
SELECT column_a, column_b
FROM table_name
WHERE condition
ORDER BY column_a DESC
LIMIT 10;
```

- `SELECT` chooses columns (`*` means all columns).
- `FROM` specifies the table.
- `WHERE` filters rows.
- `ORDER BY` sorts rows (`ASC` ascending, `DESC` descending).
- `LIMIT` caps the number of returned rows.

SQL tables have **no guaranteed order** unless you use `ORDER BY`.


In [ ]:
# London customers, oldest signup first
query('''
SELECT name, signup_date
FROM products
WHERE city = 'London'
ORDER BY signup_date ASC;
''')

,name,signup_date
0,Aisha Khan,2024-01-10
1,Chloe Smith,2024-02-18


### Exercises 1

Write queries to:
1. Show each product name and price.
2. Find shipped orders, newest first.
3. Find products costing more than £100.
4. Return the first two customers alphabetically.

Try them before opening the solutions.

In [ ]:
# Exercise 1 — write your query


first=query('''
SELECT product_name, unit_price
FROM products
ORDER BY unit_price
''')
print(first)

second=query('''
SELECT order_date, status
FROM orders
WHERE status = 'shipped'
ORDER BY order_date DESC
''')
print(second)

third=query('''
SELECT product_name, unit_price
FROM products
WHERE unit_price > 100
ORDER BY unit_price
''')
print(third)

fourth=query('''
SELECT name
FROM customers
ORDER BY name ASC
LIMIT 2
''')
print(fourth)

# fourth_solution=query('''
# SELECT name FROM customers
# ORDER BY name ASC
# LIMIT 2
# ''')
# print(fourth_solution)


  product_name  unit_price
0     Notebook         4.5
1        Mouse        25.0
2        Chair       130.0
3         Desk       220.0
4       Laptop       900.0
   order_date   status
0  2024-06-01  shipped
1  2024-05-12  shipped
2  2024-05-02  shipped
3  2024-04-03  shipped
4  2024-04-01  shipped
  product_name  unit_price
0        Chair       130.0
1         Desk       220.0
2       Laptop       900.0
         name
0  Aisha Khan
1  Ben Carter
         name
0  Aisha Khan
1  Ben Carter


<details><summary><strong>Solutions — reveal after attempting</strong></summary>

```sql
-- 1
SELECT product_name, unit_price FROM products;

-- 2
SELECT * FROM orders
WHERE status = 'shipped'
ORDER BY order_date DESC;

-- 3
SELECT product_name, unit_price FROM products
WHERE unit_price > 100;

-- 4
SELECT name FROM customers
ORDER BY name ASC
LIMIT 2;
```
</details>


## 2. Aggregation: COUNT, SUM, AVG, GROUP BY, HAVING

Aggregate functions summarise multiple rows:
- `COUNT(*)`: number of rows
- `SUM(x)`: total
- `AVG(x)`: mean
- `MIN(x)` / `MAX(x)`: minimum / maximum

`GROUP BY` creates one output row per group, much like pandas `groupby()`.

**Important distinction:** `WHERE` filters rows before aggregation; `HAVING` filters groups after aggregation. `AS` gives a result column a useful name.

In [ ]:
query('''
SELECT city, COUNT(*) AS customer_count
FROM customers
GROUP BY city
ORDER BY customer_count DESC;
''')

In [ ]:
# Average price by product category
query('''
SELECT category, AVG(unit_price) AS average_price
FROM products
GROUP BY category;
''')

### Exercises 2

1. Count orders by status.
2. Find customers who have placed at least two orders (all statuses).
3. Find the highest product price in each category.

**Hint:** `GROUP BY` the relevant category or customer, then use `HAVING COUNT(*) >= 2` for question 2.

<details><summary><strong>Solutions</strong></summary>

```sql
SELECT status, COUNT(*) AS order_count
FROM orders GROUP BY status;

SELECT customer_id, COUNT(*) AS order_count
FROM orders
GROUP BY customer_id
HAVING COUNT(*) >= 2;

SELECT category, MAX(unit_price) AS highest_price
FROM products GROUP BY category;
```
</details>


## 3. JOIN: combine related tables

Data is split across tables to reduce duplication. A join combines rows using related columns.

```sql
SELECT ...
FROM orders AS o
JOIN customers AS c
  ON o.customer_id = c.customer_id;
```

This is similar to `pd.merge(orders, customers, on="customer_id")`.

- `INNER JOIN` (or just `JOIN`) keeps matching rows.
- `LEFT JOIN` keeps every row from the left table, even without a match; missing right-side values become `NULL`.

Aliases such as `o` and `c` make queries shorter.

In [ ]:
# Orders with customer names
query('''
SELECT o.order_id, o.order_date, o.status, c.name, c.city
FROM orders AS o
JOIN customers AS c
  ON o.customer_id = c.customer_id
ORDER BY o.order_date;
''')

In [ ]:
# Include customers who have never ordered
query('''
SELECT c.name, COUNT(o.order_id) AS number_of_orders
FROM customers AS c
LEFT JOIN orders AS o
  ON c.customer_id = o.customer_id
GROUP BY c.customer_id, c.name
ORDER BY number_of_orders DESC;
''')

### A multi-table analytical query

To calculate product revenue, combine `order_items` (quantities), `products` (prices) and `orders` (status). Revenue per line is `quantity * unit_price`.

In [ ]:
query('''
SELECT p.product_name,
       SUM(oi.quantity * p.unit_price) AS revenue
FROM order_items AS oi
JOIN products AS p ON oi.product_id = p.product_id
JOIN orders AS o ON oi.order_id = o.order_id
WHERE o.status != 'cancelled'
GROUP BY p.product_id, p.product_name
ORDER BY revenue DESC;
''')

### Exercises 3

1. List each order ID, customer name and order date.
2. Calculate spending per customer, excluding cancelled orders.
3. Find customers who have never ordered.
4. Calculate revenue per product category, excluding cancelled orders.

Hint for question 2: join customers → orders → order_items → products, then sum `quantity * unit_price`.

<details><summary><strong>Solutions</strong></summary>

```sql
-- 1
SELECT o.order_id, c.name, o.order_date
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id;

-- 2
SELECT c.name, SUM(oi.quantity * p.unit_price) AS spending
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
JOIN order_items oi ON o.order_id = oi.order_id
JOIN products p ON oi.product_id = p.product_id
WHERE o.status != 'cancelled'
GROUP BY c.customer_id, c.name
ORDER BY spending DESC;

-- 3
SELECT c.name
FROM customers c
LEFT JOIN orders o ON c.customer_id = o.customer_id
WHERE o.order_id IS NULL;

-- 4
SELECT p.category, SUM(oi.quantity * p.unit_price) AS revenue
FROM order_items oi
JOIN products p ON oi.product_id = p.product_id
JOIN orders o ON oi.order_id = o.order_id
WHERE o.status != 'cancelled'
GROUP BY p.category;
```
</details>


## 4. Subqueries and CTEs

A subquery is a query inside another query. A **CTE** (common table expression) is a named intermediate result introduced with `WITH`. CTEs help organise complicated analysis into logical stages — a little like creating intermediate DataFrames in pandas.

In [ ]:
# First calculate each order's total, then filter above-average orders
query('''
WITH order_totals AS (
    SELECT o.order_id,
           SUM(oi.quantity * p.unit_price) AS total
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    JOIN products p ON oi.product_id = p.product_id
    WHERE o.status != 'cancelled'
    GROUP BY o.order_id
)
SELECT order_id, total
FROM order_totals
WHERE total > (SELECT AVG(total) FROM order_totals)
ORDER BY total DESC;
''')

### Exercises 4

1. Use a CTE to count shipped orders per customer, then return those with more than one.
2. Use a CTE to calculate revenue per category, then return categories above £100.

<details><summary><strong>Solutions</strong></summary>

```sql
WITH shipped_counts AS (
    SELECT customer_id, COUNT(*) AS n
    FROM orders WHERE status = 'shipped'
    GROUP BY customer_id
)
SELECT c.name, s.n
FROM shipped_counts s
JOIN customers c ON c.customer_id = s.customer_id
WHERE s.n > 1;

WITH category_revenue AS (
    SELECT p.category, SUM(oi.quantity * p.unit_price) AS revenue
    FROM order_items oi
    JOIN products p ON oi.product_id = p.product_id
    JOIN orders o ON oi.order_id = o.order_id
    WHERE o.status != 'cancelled'
    GROUP BY p.category
)
SELECT * FROM category_revenue WHERE revenue > 100;
```
</details>


## 5. Window functions: analyse rows without collapsing them

`GROUP BY` reduces each group to one row. A window function calculates across related rows **while retaining individual rows**.

Examples:
- `ROW_NUMBER()`: sequential row number.
- `RANK()` / `DENSE_RANK()`: rank rows, handling ties differently.
- `LAG()` / `LEAD()`: access the previous / next row.
- `SUM(x) OVER (...)`: group totals or running totals.

`PARTITION BY` creates separate windows, for example one ranking per customer.

In [ ]:
# Rank customers by spending
query('''
WITH spending AS (
    SELECT c.customer_id, c.name,
           COALESCE(SUM(
               CASE WHEN o.status != 'cancelled'
                    THEN oi.quantity * p.unit_price ELSE 0 END
           ), 0) AS total_spending
    FROM customers c
    LEFT JOIN orders o ON c.customer_id = o.customer_id
    LEFT JOIN order_items oi ON o.order_id = oi.order_id
    LEFT JOIN products p ON oi.product_id = p.product_id
    GROUP BY c.customer_id, c.name
)
SELECT name, total_spending,
       RANK() OVER (ORDER BY total_spending DESC) AS spending_rank
FROM spending
ORDER BY spending_rank;
''')

### Exercises 5

1. Number each customer's orders from oldest to newest with `ROW_NUMBER()`.
2. Show each order's value and the previous order's value with `LAG()`.
3. Rank products by revenue.

Hint: use `PARTITION BY customer_id` for an independent sequence per customer.

<details><summary><strong>Solutions</strong></summary>

```sql
-- 1
SELECT customer_id, order_id, order_date,
       ROW_NUMBER() OVER (
           PARTITION BY customer_id ORDER BY order_date
       ) AS order_number
FROM orders;

-- 2
WITH totals AS (
    SELECT o.order_id, o.order_date,
           SUM(oi.quantity * p.unit_price) AS total
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    JOIN products p ON oi.product_id = p.product_id
    GROUP BY o.order_id, o.order_date
)
SELECT order_id, order_date, total,
       LAG(total) OVER (ORDER BY order_date) AS previous_total
FROM totals;

-- 3
WITH product_revenue AS (
    SELECT p.product_name,
           SUM(oi.quantity * p.unit_price) AS revenue
    FROM products p
    JOIN order_items oi ON p.product_id = oi.product_id
    JOIN orders o ON oi.order_id = o.order_id
    WHERE o.status != 'cancelled'
    GROUP BY p.product_id, p.product_name
)
SELECT product_name, revenue,
       RANK() OVER (ORDER BY revenue DESC) AS revenue_rank
FROM product_revenue;
```
</details>


## 6. SQL from Python: parameters and pandas

In real work, Python can prepare inputs, execute SQL, and pass results to pandas.

Use placeholders rather than concatenating values into SQL strings. This is safer and handles quotes correctly. SQLite uses `?` placeholders.

In [ ]:
city_to_search = "London"

query(
    "SELECT name, signup_date FROM customers WHERE city = ? ORDER BY signup_date;",
    params=(city_to_search,)
)

### SQL result → pandas analysis

A useful rule of thumb:
- Use SQL to filter, join and aggregate data close to where it is stored.
- Use Python/pandas for statistical analysis, visualisation, modelling and workflows that benefit from Python libraries.

The boundary is flexible; choose the clearest approach.

In [ ]:
sales = query('''
SELECT o.order_date, c.city, p.category,
       oi.quantity, p.unit_price,
       oi.quantity * p.unit_price AS line_revenue
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
JOIN order_items oi ON o.order_id = oi.order_id
JOIN products p ON oi.product_id = p.product_id
WHERE o.status = 'shipped';
''')

sales.head()

In [ ]:
sales.groupby("category")["line_revenue"].sum().sort_values(ascending=False)

## 7. Mini-project: investigate the shop

Write SQL queries to answer:

1. Which city generates the most revenue from shipped orders?
2. Which customer has spent the most, excluding cancelled orders?
3. What is the average value of a shipped order?
4. Which product category has the highest revenue?
5. For each customer, how many days passed between signup and their first order?

For question 5, SQLite's `julianday(date)` function can help: subtracting two `julianday` values gives the number of days between dates.

**Stretch challenges**
- Return the top two products by revenue within each category.
- Calculate each month's shipped revenue and its month-on-month change.
- Find customers whose spending is above the average customer spending.

Write your own queries in the cells below. Check results against the source tables where possible.

In [ ]:
# Mini-project question 1
query('''
-- Write your SQL here
''')

In [ ]:
# Mini-project question 2
query('''
-- Write your SQL here
''')

In [ ]:
# Mini-project question 3
query('''
-- Write your SQL here
''')

In [ ]:
# Mini-project question 4
query('''
-- Write your SQL here
''')

In [ ]:
# Mini-project question 5
query('''
-- Write your SQL here
''')

## 8. Quick reference

| Task | SQL pattern |
|---|---|
| Select columns | `SELECT a, b FROM table` |
| Filter rows | `WHERE condition` |
| Sort | `ORDER BY column DESC` |
| Unique values | `SELECT DISTINCT column` |
| Group rows | `GROUP BY column` |
| Filter groups | `HAVING aggregate(...) condition` |
| Combine tables | `JOIN other_table ON ...` |
| Keep all left rows | `LEFT JOIN ... ON ...` |
| Test missing values | `IS NULL` |
| Replace missing values | `COALESCE(value, fallback)` |
| Conditional logic | `CASE WHEN ... THEN ... ELSE ... END` |
| Name a query stage | `WITH name AS (...)` |
| Rank rows | `RANK() OVER (ORDER BY ...)` |
| Previous row | `LAG(column) OVER (...)` |

### Common differences from pandas
- SQL results are unordered unless you specify `ORDER BY`.
- Use `IS NULL`, not `= NULL`.
- Use `AND` / `OR` for compound conditions.
- `GROUP BY` collapses rows; window functions preserve them.
- SQL dialects differ slightly. These concepts transfer to PostgreSQL, MySQL and SQL Server, but check syntax when switching.

### Suggested next steps
1. Practise on a realistic dataset.
2. Learn PostgreSQL and how to connect to a database.
3. Learn indexes and query plans.
4. Build confidence with date/time functions and more advanced window functions.

**Portfolio tip:** add your completed mini-project answers, a README describing the learning objectives, and a note that this is a fictional practice dataset before publishing to GitHub.
